# DelayShield 01 - Exploratory Data Analysis

Inspection of the raw Olist tables, the delay target and the leakage risks.
Everything here reuses the production modules in `src/` - the notebook is a
window onto the pipeline, not a parallel implementation.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import data_loader as dl
from preprocessing import build_order_base
pd.set_option('display.width', 140)

## 1. Raw tables: rows, columns, missing values

In [ ]:
dl.dataset_profile()

## 2. Orders table schema and status mix

In [ ]:
orders = dl.load_table('orders')
display(orders.head(3))
display(orders['order_status'].value_counts())
display(orders.isna().mean().round(4).to_frame('missing_share'))

## 3. Build the delivered-order base table

`build_order_base()` filters to delivered orders with all required timestamps,
constructs the target, aggregates order items to one row per order (keeping
`seller_count` and a primary seller), joins customers/sellers/geo/reviews and
computes the great-circle distance proxy.

In [ ]:
base = build_order_base()
print(base.shape)
print(base['order_purchase_timestamp'].min(), '->', base['order_purchase_timestamp'].max())
base[['order_id','primary_seller_id','seller_count','total_order_value',
      'total_freight_value','distance_km','delivery_days','delay_days','delivery_delay']].head()

## 4. Target distribution

In [ ]:
print('late share:', round(base['delivery_delay'].mean(), 4))
print(base['delivery_delay'].value_counts())
base['delivery_delay'].value_counts().plot(kind='bar', color=['#1a9d5a','#d1394a'])
plt.title('On time (0) vs late (1)'); plt.show()

### Delay rate is not stationary - this is why the split is chronological

In [ ]:
monthly = base.groupby(base['order_purchase_timestamp'].dt.to_period('M')).agg(
    orders=('order_id','size'), delay_rate=('delivery_delay','mean'))
monthly['delay_rate'].plot(figsize=(11,4), marker='o', color='#d1394a')
plt.title('Monthly delay rate'); plt.ylabel('delay rate'); plt.grid(alpha=.3); plt.show()
monthly.tail(12)

## 5. Multi-seller orders

In [ ]:
vc = base['seller_count'].value_counts().sort_index()
print(vc)
print('multi-seller share:', round((base['seller_count'] > 1).mean(), 4))

## 6. Where do delays concentrate?

In [ ]:
cat = (base.groupby('product_category')
       .agg(orders=('order_id','size'), delay_rate=('delivery_delay','mean'))
       .query('orders >= 500').sort_values('delay_rate', ascending=False))
display(cat.head(10))

state = (base.groupby('customer_state')
         .agg(orders=('order_id','size'), delay_rate=('delivery_delay','mean'))
         .query('orders >= 300').sort_values('delay_rate', ascending=False))
display(state.head(10))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,4))
base.boxplot(column='distance_km', by='delivery_delay', ax=ax[0])
ax[0].set_title('Distance vs delay'); ax[0].set_ylim(0, 3000)
base.boxplot(column='total_freight_value', by='delivery_delay', ax=ax[1])
ax[1].set_title('Freight vs delay'); ax[1].set_ylim(0, 100)
plt.suptitle(''); plt.show()

## 7. Leakage risks identified

| Column | Risk | Handling |
|---|---|---|
| `order_delivered_customer_date` | defines the target | target only, never a feature |
| `order_delivered_carrier_date` | post-confirmation | excluded |
| `order_approved_at` | post-confirmation | excluded |
| `review_score` | post-delivery | supplier scorecard only |
| `order_status` | post-delivery | filter only |
| seller statistics over the full dataset | look-ahead | replaced by lagged statistics using only orders already delivered before the current purchase timestamp |

The full report is regenerated by `python src/eda_report.py` into
`reports/eda_report.md`.